# LSTM Patient Deterioration Warning
## Can a neural network remember an important warning from hours ago?

> **Educational demo only.** This project uses synthetic data and is not a clinical prediction tool.

We will use **24 hours of vital signs** to predict whether a patient belongs to a deterioration pattern.

```text
24 hours × 3 vital signs → RNN / LSTM → deterioration risk
```

Vital signs: **heart rate, respiratory rate, SpO₂**.


# 1. Load the dataset

The dataset is already stored in this GitHub repository. We simply download it with Pandas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

DATA_URL = 'https://raw.githubusercontent.com/dujing82-blip/tiny-lstm-patient-deterioration/main/patient_vital_signs.csv'
data = pd.read_csv(DATA_URL)

print(data.head())
print('\nRows:', len(data))
print('Patients:', data['patient_id'].nunique())


# 2. Look at one patient's 24-hour history

A sequence model does not see just one measurement. It sees how the measurements **change over time**.


In [ ]:
patient = data[data['patient_id'] == 750]

plt.figure(figsize=(12,4))
plt.plot(patient['hour'], patient['heart_rate_bpm'], marker='o', label='Heart rate')
plt.plot(patient['hour'], patient['respiratory_rate_bpm'], marker='o', label='Respiratory rate')
plt.plot(patient['hour'], patient['spo2_percent'], marker='o', label='SpO₂')
plt.xlabel('Hour')
plt.ylabel('Measurement')
plt.title('One Patient: 24 Hours of Vital Signs')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print('Case type:', patient['case_type'].iloc[0])
print('Deterioration label:', patient['deterioration'].iloc[0])


# 3. Turn the table into RNN sequences

Each patient becomes one training example:

```text
24 time steps
     ×
3 features at every time step

X.shape = (patients, 24, 3)
```


In [ ]:
FEATURES = ['heart_rate_bpm', 'respiratory_rate_bpm', 'spo2_percent']

X = []
y = []
case_types = []

for patient_id, patient in data.groupby('patient_id'):
    X.append(patient[FEATURES].to_numpy())
    y.append(patient['deterioration'].iloc[0])
    case_types.append(patient['case_type'].iloc[0])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)
case_types = np.array(case_types)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nOne patient =', X[0].shape, '= 24 hours × 3 vital signs')


# 4. Train / test split and normalization

We keep 20% of the patients completely unseen for testing. We calculate normalization values using the training patients only.


In [ ]:
# Shuffle PATIENTS, not individual hours.
rng = np.random.default_rng(7)
indices = rng.permutation(len(X))
split = int(0.8 * len(X))
train_idx = indices[:split]
test_idx = indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
case_test = case_types[test_idx]

# Mean and standard deviation for each of the 3 vital signs.
mean = X_train.mean(axis=(0,1), keepdims=True)
std = X_train.std(axis=(0,1), keepdims=True)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print('Training patients:', len(X_train))
print('Testing patients:', len(X_test))


# 5. First try: SimpleRNN

This is the model from our previous lesson. It reads the 24 hours in order and carries a hidden state forward.


In [ ]:
rnn_model = keras.Sequential([
    layers.Input(shape=(24,3)),
    layers.SimpleRNN(16),
    layers.Dense(1, activation='sigmoid')
])

rnn_model.compile(
    optimizer='adam',
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

rnn_model.summary()

rnn_model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


# 6. Now replace RNN with LSTM

Look carefully: the model is almost identical. We change essentially **one layer**.

```text
SimpleRNN(16)  →  LSTM(16)
```

LSTM was designed to preserve useful information over longer sequences.


In [ ]:
lstm_model = keras.Sequential([
    layers.Input(shape=(24,3)),
    layers.LSTM(16),                 # <-- the important change
    layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

lstm_model.summary()

lstm_model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


# 7. Compare RNN and LSTM


In [ ]:
rnn_prob = rnn_model.predict(X_test, verbose=0).flatten()
lstm_prob = lstm_model.predict(X_test, verbose=0).flatten()

rnn_pred = (rnn_prob >= 0.5).astype(int)
lstm_pred = (lstm_prob >= 0.5).astype(int)

rnn_accuracy = np.mean(rnn_pred == y_test)
lstm_accuracy = np.mean(lstm_pred == y_test)

print(f'SimpleRNN overall accuracy: {rnn_accuracy:.3f}')
print(f'LSTM overall accuracy:      {lstm_accuracy:.3f}')


# 8. The important test: early-warning patients

Some deterioration examples contain an important abnormal event **early** in the 24-hour history, followed by partial recovery.

These cases ask the model to remember information that is farther in the past.


In [ ]:
early = (case_test == 'early_warning')

rnn_early_accuracy = np.mean(rnn_pred[early] == y_test[early])
lstm_early_accuracy = np.mean(lstm_pred[early] == y_test[early])

print('Number of early-warning test patients:', early.sum())
print(f'SimpleRNN accuracy on early-warning cases: {rnn_early_accuracy:.3f}')
print(f'LSTM accuracy on early-warning cases:      {lstm_early_accuracy:.3f}')


# 9. Visual comparison


In [ ]:
labels = ['Overall', 'Early warning']
rnn_scores = [rnn_accuracy, rnn_early_accuracy]
lstm_scores = [lstm_accuracy, lstm_early_accuracy]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8,5))
plt.bar(x - width/2, rnn_scores, width, label='SimpleRNN')
plt.bar(x + width/2, lstm_scores, width, label='LSTM')
plt.xticks(x, labels)
plt.ylim(0,1.05)
plt.ylabel('Accuracy')
plt.title('SimpleRNN vs. LSTM')
plt.legend()
plt.show()


# 10. Why LSTM?

The key question is not simply **“Which model has higher accuracy?”**

The deeper question is:

> **Can the network preserve important information when it happened many time steps ago?**

```text
Hour 4: important warning
       ↓
       ───────── many later measurements ─────────→ Hour 24
                                                    ↓
                                               prediction
```

A SimpleRNN carries a hidden state, but information can become difficult to preserve over long sequences. LSTM adds a **cell state and gates** that control what information should be remembered, updated, or forgotten.

That is why our next question is: **How do the LSTM gates work?**

> Reminder: this synthetic classroom exercise demonstrates sequence-model concepts only. It must not be interpreted as a clinical deterioration score or used for patient care.
